# Knowledge Mindmap

This notebook builds a **knowledge distance map** centred on one Wikipedia article.

Given a centre page A and a list of target pages [B, C, D, …], we:
1. Run BFS from A to each target to find the shortest path
2. Record hop-distances: how many clicks does it take to get from A to each target?
3. Compare distances — some targets will be surprisingly close, others far
4. Visualise everything as a radial map where distance from A = conceptual distance

This answers: **"How far is everything from what I care about?"**

> Example: Starting from "Data science", which is closer — "Machine learning" or "Ancient Rome"?  
> Intuitively ML should be closer. But by how many clicks? And what's the path?

In [ ]:
import sys
sys.path.insert(0, "..")

from src import WikiMindmap, WikiScraper

## Configuration

Change `CENTER` and `TARGETS` to explore any knowledge domain.  
`MAX_PAGES` is the BFS search cap per target — higher means more thorough but slower.

In [ ]:
CENTER = "Data science"

TARGETS = [
    "Machine learning",
    "Artificial intelligence",
    "Statistics",
    "Ancient Rome",
    "Dinosaur",
    "Philosophy",
    "William Shakespeare",
]

MAX_PAGES = 200   # BFS page limit per target

scraper = WikiScraper(delay=0.3)

## Build the mindmap

For each target, BFS explores from the centre outward until it finds the target or hits the page limit.

In [ ]:
mm = WikiMindmap(
    center=CENTER,
    scraper=scraper,
    max_pages=MAX_PAGES,
    verbose=True,
)

mm.build(TARGETS)

## Distance summary table

In [ ]:
mm.summary()

## Quick stats

In [ ]:
closest  = mm.closest()
farthest = mm.farthest()

if closest:
    print(f"Closest  target: '{closest.target}'  ({closest.hops} hops)")
    print(f"  Path: {' → '.join(closest.path)}")
if farthest:
    print(f"\nFarthest target: '{farthest.target}'  ({farthest.hops} hops)")
    print(f"  Path: {' → '.join(farthest.path)}")

print("\nAll distances:")
for target, hops in sorted(mm.distances().items(), key=lambda x: (x[1] is None, x[1])):
    bar = "█" * (hops or 0)
    print(f"  {target:<40} {str(hops):>4} hops  {bar}")

## Visualise the knowledge map

In [ ]:
mm.visualize(figsize=(15, 11), save_path="mindmap.png")

## Interpretation

The visualisation shows:
- **Red node** — the centre article (your starting point)
- **Green nodes** — target articles with their hop counts
- **Blue nodes** — intermediate pages on each shortest path
- **Arrows** — the direction links were followed

Targets with fewer hops are conceptually closer in Wikipedia's link structure.  
Targets many hops away require passing through more unrelated topics to reach.

## Try your own topics

In [ ]:
# Change center and targets to explore a completely different knowledge domain
mm2 = WikiMindmap(
    center="Python (programming language)",
    scraper=scraper,
    max_pages=200,
    verbose=False,
)
mm2.build([
    "Machine learning",
    "Web scraping",
    "Ancient Greece",
    "Monty Python",
    "Snake",
])
mm2.summary()
mm2.visualize(save_path="mindmap_python.png")